# Moving Average Filter

**Dataset**: PhysioNet Auditory EEG (Abo Alzahab et al., 2021)  
**Channels**: P4, Cz, F8, T7  
**Sampling rate**: 200 Hz  
**Subject**: 1

---

## Overview

The moving average filter computes each sample as the average of its neighboring samples. Larger windows produce more smoothing and remove more noise, but the signal loses fine details.

## What you should expect to see

- Small window (5): light smoothing, noise still visible
- Medium window (11): moderate smoothing, fine details fade
- Large window (21): strong smoothing, signal becomes very smooth and loses fast features

## Key parameters

| Parameter | Value | Meaning |
| --- | --- | --- |
| Channel | P4 | Parietal region |
| Sampling rate | 200 Hz | One sample every 5 ms |
| Windows | 5, 11, 21 | Window sizes |
| Plotted samples | 5000 | First 25 seconds |


## 1. Install dependencies


In [ ]:
!pip install scipy numpy plotly wfdb


## 2. Clone the resources repo and download one subject

We download only one subject (`--subjects 1`) to speed up the experiment in Colab.


In [ ]:
import os
if not os.path.exists('python-EEG-Arabic-Resources'):
    !git clone https://github.com/NibrasAz7/python-EEG-Arabic-Resources.git
os.chdir('python-EEG-Arabic-Resources')


In [ ]:
from pathlib import Path
data_dir = Path('data/local')
if not data_dir.exists() or not any(data_dir.glob('*.dat')):
    !python data/download_local.py --output data/local --subjects 1


## 3. Load the EEG signal

We load subject 1, experiment 1, session 2, channel **P4** (parietal region).


In [ ]:
import numpy as np
from utils.eeg_loader import load_local_eeg

timestamps, eeg_data, ch_names = load_local_eeg(
    data_dir='data/local', subject=1, experiment=1, session=2
)
channel_data = eeg_data[:, 0]  # P4 channel
fs = 200  # Sampling rate (Hz)

print(f'Channels: {ch_names}')
print(f'Signal length: {len(channel_data)} samples ({len(channel_data)/fs:.1f} seconds)')


## 4. Apply the filter

We use `np.convolve` with `mode='same'` to apply the moving average. The kernel is a vector of ones divided by the window size.


In [ ]:
windows = [5, 11, 21]
filtered = {}
for w in windows:
    kernel = np.ones(w) / w
    filtered[w] = np.convolve(channel_data, kernel, mode='same')
print(f'Applied moving average with windows: {windows}')


## 5. Interactive plot

**What to look for:**

- Window 5: noise still visible
- Window 11: fast noise removed, details preserved
- Window 21: strong smoothing hides fast features
- Use the zoom tool to inspect specific time ranges


In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

n_plot = min(5000, len(channel_data))
t_sec = timestamps[:n_plot] / 1000.0

fig = make_subplots(rows=4, cols=1, shared_xaxes=True,
                    subplot_titles=('Original (P4)',
                                    'Moving average (window=5)',
                                    'Moving average (window=11)',
                                    'Moving average (window=21)'))
fig.add_trace(go.Scatter(x=t_sec, y=channel_data[:n_plot], name='Raw',
                         line=dict(color='gray', width=0.5)), row=1, col=1)
for i, w in enumerate(windows, start=2):
    fig.add_trace(go.Scatter(x=t_sec, y=filtered[w][:n_plot],
                             name=f'w={w}', line=dict(width=0.5)), row=i, col=1)
fig.update_layout(height=900, title_text='Moving Average Filter - Channel P4',
                  xaxis4_title='Time (s)', showlegend=False)
fig.show()


## What did we learn?

- The moving average reduces noise by averaging neighboring samples
- Larger windows produce stronger smoothing but hide details
- Smaller windows preserve details but do not fully remove noise
- The researcher chooses the window size to balance smoothing and detail retention
